# E-comers 360 | Evolución del negocio

## Pregunta de negocio

¿Cómo evolucionan las órdenes, el valor vendido, el flete y la satisfacción en el tiempo?

In [2]:
paquetes <- c(
  "readr", "dplyr", "tidyr", "stringr",
  "lubridate", "ggplot2", "purrr", "scales"
)

por_instalar <- paquetes[
  !vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)
]

if (length(por_instalar) > 0) {
  install.packages(por_instalar, repos = "https://cloud.r-project.org")
}

invisible(lapply(paquetes, library, character.only = TRUE))

options(scipen = 999)

In [3]:
ruta_datos <- "../data/olist"

orders <- read_csv(
  file.path(ruta_datos, "olist_orders_dataset.csv"),
  show_col_types = FALSE
)

items <- read_csv(
  file.path(ruta_datos, "olist_order_items_dataset.csv"),
  show_col_types = FALSE
)

reviews <- read_csv(
  file.path(ruta_datos, "olist_order_reviews_dataset.csv"),
  show_col_types = FALSE
)

In [ ]:
# Inspección inicial

dim(orders)
dim(items)
dim(reviews)

names(orders)
names(items)
names(reviews)

[1] 99441     8

[1] 112650      7

[1] 99224     7

[1] "order_id"                      "customer_id"                  
[3] "order_status"                  "order_purchase_timestamp"     
[5] "order_approved_at"             "order_delivered_carrier_date" 
[7] "order_delivered_customer_date" "order_estimated_delivery_date"

[1] "order_id"            "order_item_id"       "product_id"         
[4] "seller_id"           "shipping_limit_date" "price"              
[7] "freight_value"

[1] "review_id"               "order_id"               
[3] "review_score"            "review_comment_title"   
[5] "review_comment_message"  "review_creation_date"   
[7] "review_answer_timestamp"

In [ ]:
# Tipos de las columnas relevantes
glimpse(orders)
glimpse(items)
glimpse(reviews)

# órdenes únicas hay en cada tabla
orders %>%
  summarise(
    filas = n(),
    ordenes_unicas = n_distinct(order_id),
    duplicados_order_id = n() - n_distinct(order_id)
  )

items %>%
  summarise(
    filas = n(),
    ordenes_unicas = n_distinct(order_id),
    duplicados_order_id = n() - n_distinct(order_id)
  )

reviews %>%
  summarise(
    filas = n(),
    ordenes_unicas = n_distinct(order_id),
    duplicados_order_id = n() - n_distinct(order_id)
  )

Rows: 99,441
Columns: 8
$ order_id                      <chr> "e481f51cbdc54678b7cc49136f2d6af7", "53c…
$ customer_id                   <chr> "9ef432eb6251297304e76186b10a928d", "b08…
$ order_status                  <chr> "delivered", "delivered", "delivered", "…
$ order_purchase_timestamp      <dttm> 2017-10-02 10:56:33, 2018-07-24 20:41:3…
$ order_approved_at             <dttm> 2017-10-02 11:07:15, 2018-07-26 03:24:2…
$ order_delivered_carrier_date  <dttm> 2017-10-04 19:55:00, 2018-07-26 14:31:0…
$ order_delivered_customer_date <dttm> 2017-10-10 21:25:13, 2018-08-07 15:27:4…
$ order_estimated_delivery_date <dttm> 2017-10-18, 2018-08-13, 2018-09-04, 201…


Rows: 112,650
Columns: 7
$ order_id            <chr> "00010242fe8c5a6d1ba2dd792cb16214", "00018f77f2f03…
$ order_item_id       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1,…
$ product_id          <chr> "4244733e06e7ecb4970a6e2683c13e61", "e5f2d52b80218…
$ seller_id           <chr> "48436dade18ac8b2bce089ec2a041202", "dd7ddc04e1b6c…
$ shipping_limit_date <dttm> 2017-09-19 09:45:35, 2017-05-03 11:05:13, 2018-01…
$ price               <dbl> 58.90, 239.90, 199.00, 12.99, 199.90, 21.90, 19.90…
$ freight_value       <dbl> 13.29, 19.93, 17.87, 12.79, 18.14, 12.69, 11.85, 7…
Rows: 99,224
Columns: 7
$ review_id               <chr> "7bc2406110b926393aa56f80a40eba40", "80e641a11…
$ order_id                <chr> "73fc7af87114b39712e6da79b0a377eb", "a548910a1…
$ review_score            <dbl> 4, 5, 5, 5, 5, 1, 5, 5, 5, 4, 5, 5, 4, 4, 3, 5…
$ review_comment_title    <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, "recomendo…
$ review_comment_message  <chr> NA, NA, NA, "Recebi bem antes do prazo 

filas,ordenes_unicas,duplicados_order_id
<int>,<int>,<int>
99441,99441,0


filas,ordenes_unicas,duplicados_order_id
<int>,<int>,<int>
112650,98666,13984


filas,ordenes_unicas,duplicados_order_id
<int>,<int>,<int>
99224,98673,551


In [ ]:
# Revisar multiplicidad de reseñas por orden

reviews %>%
  count(order_id, name = "n_reviews") %>%
  count(n_reviews, name = "n_ordenes") %>%
  arrange(n_reviews)

reviews %>%
  count(order_id, name = "n_reviews") %>%
  filter(n_reviews > 1) %>%
  arrange(desc(n_reviews)) %>%
  head(10)

n_reviews,n_ordenes
<int>,<int>
1,98126
2,543
3,4


order_id,n_reviews
<chr>,<int>
03c939fd7fd3b38f8485a0f95798f1f6,3
8e17072ec97ce29f0e1f111e598b0c85,3
c88b1d1b157a9999ce368f218a407141,3
df56136b8031ecd28e200bb18e6ddb2e,3
0035246a40f520710769010f752e7507,2
013056cfe49763c6f66bda03396c5ee3,2
0176a6846bcb3b0d3aa3116a9a768597,2
02355020fd0a40a0d56df9f6ff060413,2
029863af4b968de1e5d6a82782e662f5,2


In [ ]:
# reseñas múltiples por orden

sum(duplicated(reviews))
sum(duplicated(reviews$review_id))

reviews %>%
  add_count(order_id, name = "n_reviews") %>%
  filter(n_reviews > 1) %>%
  arrange(order_id, review_creation_date) %>%
  select(
    order_id,
    review_id,
    review_score,
    review_creation_date,
    review_answer_timestamp
  ) %>%
  head(20)

[1] 0

[1] 814

order_id,review_id,review_score,review_creation_date,review_answer_timestamp
<chr>,<chr>,<dbl>,<dttm>,<dttm>
0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-25,2017-08-29 21:45:57
0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-29,2017-08-30 01:59:12
013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22,2018-02-23 12:12:30
013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04,2018-03-05 17:02:00
0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2017-12-30,2018-01-02 10:54:06
0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2017-12-30,2018-01-02 10:54:47
02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-21,2018-03-22 01:32:08
02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-29,2018-03-30 03:16:19
029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-14,2017-07-17 13:58:06


In [ ]:
# Verificación 2: valores nulos

orders %>%
  summarise(
    fecha_compra_na = sum(is.na(order_purchase_timestamp))
  )

items %>%
  summarise(
    price_na = sum(is.na(price)),
    freight_na = sum(is.na(freight_value))
  )

reviews %>%
  summarise(
    review_score_na = sum(is.na(review_score))
  )

fecha_compra_na
<int>
0


price_na,freight_na
<int>,<int>
0,0


review_score_na
<int>
0


In [ ]:
# agregación de items por orden

items_orden <- items %>%
  group_by(order_id) %>%
  summarise(
    valor_vendido = sum(price),
    flete = sum(freight_value),
    n_items = n(),
    .groups = "drop"
  )

items_orden %>%
  summarise(
    filas = n(),
    ordenes_unicas = n_distinct(order_id)
  )

all.equal(sum(items$price), sum(items_orden$valor_vendido))
all.equal(sum(items$freight_value), sum(items_orden$flete))

filas,ordenes_unicas
<int>,<int>
98666,98666


[1] TRUE

[1] TRUE

In [ ]:
# Resumir reseñas a nivel de orden 

reviews_orden <- reviews %>%
  group_by(order_id) %>%
  summarise(
    satisfaccion = mean(review_score),
    n_reviews = n(),
    .groups = "drop"
  )

reviews_orden %>%
  summarise(
    filas = n(),
    ordenes_unicas = n_distinct(order_id)
  )

filas,ordenes_unicas
<int>,<int>
98673,98673
